# G13 - the rank wall, checked directly

G11 and G12 establish WHERE the cliff is. They do not measure WHY. Four direct checks, each able to falsify on its own: the source spans only ~768 of 1024 hub directions; the head has under 5% of its weight mass beyond 768; the wider encoders put real signal there; and **zeroing exactly those directions restores transfer**.

The ablation is the decisive one - nothing but this account predicts that deleting the unread region undoes the collapse.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G13 — the rank wall, checked directly rather than inferred from location.
# Run after G12. Requires the same caches; rebuilds the hub itself.
#
# WHAT IS ALREADY ESTABLISHED, AND WHAT IS NOT. G11 and G12 show WHERE the
# cliff is: at the source encoder's ambient dimension, for three sources,
# under two head targets, on a grid with three widths that are nobody's
# dimension and show nothing. Six curves, six hits, predicted in advance.
#
# That is strong evidence of a LOCATION. It is not a measurement of a
# MECHANISM. The rank-wall story says the cliff happens because a
# d-dimensional source's entry map cannot span more than d hub directions,
# so the head fitted on its coordinates has never seen the rest. Every
# result so far is consistent with that - and would also be consistent
# with any other account that happens to break at the same width.
#
# This notebook measures the mechanism itself. Four checks, each of which
# could come back against it:
#
#   1  RANK. The effective rank of img_small's hub coordinates at width
#      1024 should be about 768, not 1024. If it is 1024 the premise is
#      simply false and everything above is coincidence.
#
#   2  WHERE THE HEAD LOOKS. The head is fitted on those coordinates, so
#      its weight mass should sit almost entirely inside the first ~768
#      hub directions and be near zero beyond. If the head has substantial
#      mass past 768, it is not blind to those directions after all.
#
#   3  WHAT THE OTHER ENCODERS PUT THERE. A wider encoder should populate
#      the directions past 768 with real signal. If those directions are
#      empty for everyone, nothing is being misread and the mechanism has
#      nothing to act on.
#
#   4  THE DECISIVE ONE - SURGICAL ABLATION. Zero the other encoders' hub
#      coordinates beyond width 768 and re-measure transfer at 1024. The
#      account says this RESTORES transfer, because the head is no longer
#      shown directions it cannot read. Nothing else predicts that. If
#      transfer stays collapsed, the rank wall is the wrong explanation
#      even though it predicts the right location.
#
# PRE-REGISTERED: rank ~768 (within 5 per cent), head mass beyond 768
# under 5 per cent of total, other encoders' mass beyond 768 above 10 per
# cent, and ablation recovering transfer to within 10 points of its
# width-512 value. Check 4 is the one that matters; the first three are
# necessary conditions that would each falsify the account on their own.
# ==========================================================
import os
import numpy as np
from pathlib import Path

DATA_DIR = Path(os.environ["DATA_DIR"])
ALPHA, N_EVAL, SEED = 1e-2, 1000, 0
SOURCE, PROBE_WIDTH = "img_small", 1024      # 1024 > 768, well past the wall
BASE_WIDTH = 512                              # a known-good width, for reference

SPACES = {}
for size in ("small", "base", "large"):
    SPACES[f"img_{size}"] = np.load(
        str(DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz")
    )["img"].astype(np.float64)
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
SPACES["txt_bge"] = np.load(
    str(DATA_DIR / "crossmodal_pairs.npz"))["txt"].astype(np.float64)[:N]
SOURCES = ["img_small", "img_base", "img_large"]
D_SRC = SPACES[SOURCE].shape[1]
print(f"source {SOURCE} is {D_SRC}-d; probing the hub at {PROBE_WIDTH}\n")

rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
te, tr = perm[:N_EVAL], perm[N_EVAL:]


def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)


def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)


def r1(P, G):
    return float(((l2n(P) @ l2n(G).T).argmax(1) == np.arange(len(P))).mean())


def eff_rank(M):
    """Entropy-based effective rank: exp of the entropy of the normalised
    eigenvalue spectrum. Reported instead of a hard rank because a ridge
    fit leaves tiny non-zero singular values rather than exact zeros."""
    s = np.linalg.svd(M - M.mean(0), compute_uv=False)
    p = s ** 2 / (s ** 2).sum()
    p = p[p > 1e-15]
    return float(np.exp(-(p * np.log(p)).sum()))


_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_u, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)


def hub_at(d):
    B = _VT[:d].T / (_sv[:d] / np.sqrt(len(_ref)))
    return (_ref - _mu) @ B


T = SPACES["txt_bge"]
gal = l2n(T[te])
H = hub_at(PROBE_WIDTH)
to_hub = {k: ridge(SPACES[k][tr], H) for k in SOURCES}
head = ridge(SPACES[SOURCE][tr] @ to_hub[SOURCE], T[tr])

In [ ]:
# ---------- 1. rank ----------
print("=" * 72)
print("CHECK 1  effective rank of each encoder's hub coordinates")
print("=" * 72)
ranks = {}
for enc in SOURCES:
    C = SPACES[enc][tr] @ to_hub[enc]
    ranks[enc] = eff_rank(C)
    print(f"  {enc:11s} ambient {SPACES[enc].shape[1]:5d}   "
          f"hub coords {C.shape[1]}-d   effective rank {ranks[enc]:7.1f}")
r_ok = ranks[SOURCE] < 1.05 * D_SRC
print(f"\n  {SOURCE} reaches {ranks[SOURCE]:.0f} of {PROBE_WIDTH} hub "
      f"directions - predicted about {D_SRC}")
print("  " + ("PREMISE HOLDS: the source cannot span the full hub."
               if r_ok else
               "PREMISE FAILS: the source spans more than its own dimension, "
               "so the rank argument is wrong at the first step."))

In [ ]:
# ---------- 2. where the head's mass sits ----------
print("\n" + "=" * 72)
print("CHECK 2  head weight mass by hub direction")
print("=" * 72)
mass = (head ** 2).sum(axis=1)
mass = mass / mass.sum()
inside = float(mass[:D_SRC].sum())
beyond = float(mass[D_SRC:].sum())
print(f"  directions 0-{D_SRC-1}:   {inside:6.1%} of head weight mass")
print(f"  directions {D_SRC}-{PROBE_WIDTH-1}: {beyond:6.1%}")
h_ok = beyond < 0.05
print("  " + ("HEAD IS BLIND past the source's dimension, as predicted."
               if h_ok else
               "HEAD HAS REAL MASS beyond the wall - it is not blind, so the "
               "mechanism is not what the account describes."))

In [ ]:
# ---------- 3. what the wider encoders put there ----------
print("\n" + "=" * 72)
print("CHECK 3  signal beyond the wall, per encoder")
print("=" * 72)
for enc in SOURCES:
    C = SPACES[enc][te] @ to_hub[enc]
    v = C.var(axis=0)
    frac = float(v[D_SRC:].sum() / v.sum())
    print(f"  {enc:11s} {frac:6.1%} of its hub variance sits beyond "
          f"direction {D_SRC}")
others = [SPACES[e][te] @ to_hub[e] for e in SOURCES if e != SOURCE]
o_frac = float(np.mean([c.var(0)[D_SRC:].sum() / c.var(0).sum() for c in others]))
o_ok = o_frac > 0.10
print("  " + ("THE UNREAD REGION IS OCCUPIED: wider encoders put real signal "
               "where the head cannot look."
               if o_ok else
               "THE REGION IS NEARLY EMPTY, so there is little for the head "
               "to misread and the mechanism has nothing to act on."))

In [ ]:
# ---------- 4. surgical ablation, the decisive test ----------
print("\n" + "=" * 72)
print("CHECK 4  ablation - zero the unread directions and re-measure")
print("=" * 72)


def transfer(width, truncate_at=None):
    Hw = hub_at(width)
    th = {k: ridge(SPACES[k][tr], Hw) for k in SOURCES}
    hd = ridge(SPACES[SOURCE][tr] @ th[SOURCE], T[tr])
    out = []
    for enc in SOURCES:
        if enc == SOURCE:
            continue
        C = SPACES[enc][te] @ th[enc]
        if truncate_at is not None:
            C = C.copy()
            C[:, truncate_at:] = 0.0
        zero = r1(C @ hd, gal)
        nat = r1(SPACES[enc][te] @ ridge(SPACES[enc][tr], T[tr]), gal)
        out.append(zero / max(nat, 1e-9))
    return float(np.mean(out))


base = transfer(BASE_WIDTH)
broken = transfer(PROBE_WIDTH)
fixed = transfer(PROBE_WIDTH, truncate_at=D_SRC)
print(f"  width {BASE_WIDTH}, below the wall               {base:7.3f}")
print(f"  width {PROBE_WIDTH}, above the wall              {broken:7.3f}")
print(f"  width {PROBE_WIDTH}, directions >{D_SRC} zeroed    {fixed:7.3f}")
recovered = fixed - broken
a_ok = fixed > base - 0.10
print(f"\n  recovery from ablation: {recovered:+.3f}")
print("  " + ("ABLATION RESTORES TRANSFER. Removing exactly the directions "
               "the head cannot read undoes the collapse."
               if a_ok else
               "ABLATION DOES NOT RESTORE TRANSFER. The unread directions "
               "are not what breaks it, so the account predicts the right "
               "location for the wrong reason."))

In [ ]:
# ---------- verdict ----------
print("\n" + "=" * 72)
passed = [r_ok, h_ok, o_ok, a_ok]
names = ["rank premise", "head blindness", "region occupied", "ablation"]
for n, ok in zip(names, passed):
    print(f"  {'PASS' if ok else 'FAIL'}  {n}")
print()
if all(passed):
    print("MECHANISM CONFIRMED DIRECTLY, not inferred from where the cliff")
    print("falls. The source spans only its own dimension; the head's weight")
    print("mass stops there; the wider encoders put real signal past it; and")
    print("deleting exactly that signal restores transfer.")
    print()
    print("The consequence for the report is larger than the cliff itself.")
    print("512 was never a tuned operating point - any width up to the")
    print("source's dimension works, and nothing above it can. The rule is")
    print("hub_width <= dim(source encoder), and the portability-versus-")
    print("losslessness trade-off in C.13 is not a property of shared linear")
    print("coordinates but a consequence of training the head on the")
    print("NARROWEST encoder in the set. Train it on img_large and the")
    print("ceiling is 2048.")
elif a_ok and not all(passed):
    print("ABLATION PASSES but a necessary condition does not. The causal")
    print("test is the strong one, so the account is probably right and the")
    print("failing check needs explaining - report both rather than picking.")
else:
    print("MECHANISM NOT CONFIRMED. The location results in G11 and G12")
    print("stand as measured - the cliff really does sit at the source's")
    print("dimension - but the reason offered for it does not survive direct")
    print("measurement. Report the location, withdraw the mechanism, and add")
    print("it to the ledger as the eighth falsified account.")

print("\nScope: one source encoder probed at one width above its wall, one")
print("hub protocol, one corpus. The ablation is a surgical intervention on")
print("held-out coordinates, not a retrained model.")